# Module 2 · Lesson 01: Zero-Shot Prompting

**Zero-shot prompting** means asking the model to perform a task *without any examples*.
The model relies entirely on its training data and your instructions.

## What you will learn
1. Effective zero-shot prompt patterns
2. Output **format specification**
3. **Role/persona** assignment
4. Zero-shot **classification**
5. **Structured output** (JSON) extraction
6. Using **constraints** to control output

In [3]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")
 

OpenAI client ready - using model gpt-4o-mini


In [2]:
def ask(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 1200, ai_model: str="gpt-4o-mini"):
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

---
## 1. Simple Zero-Shot

The simplest form — just ask directly:

In [5]:
prompt = "What is the capital of France?"
result = ask(prompt)
display(Markdown(f"**Q**: {prompt}\n\n**A**: {result}"))

**Q**: What is the capital of France?

**A**: The capital of France is Paris.

---
## 2. Format Specification

Tell the model **exactly** what format you want:

In [6]:
prompt = """Extract the email address from this text:

Hi, you can reach me at john.doe@example.com for more information."""

result = ask(prompt, max_resp_tokens=50)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `The email address extracted from the text is: john.doe@example.com.`

In [7]:
prompt = """Extract the email address from this text and return ONLY the email, nothing else:

Hi, you can reach me at john.doe@example.com for more information."""

result = ask(prompt, max_resp_tokens=50)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `john.doe@example.com`

---
## 3. Role / Persona Assignment

The system prompt sets *who* the model should be:

In [8]:
prompt = "Explain what an API is to a non-technical person."
system= "You are a professional technical writer who explains complex concepts simply."

result = ask(prompt, system_content= system)

display(Markdown(f"### Technical Writer\n\n{result}"))

### Technical Writer

Sure! An API, or Application Programming Interface, is like a waiter at a restaurant. When you go to a restaurant, you look at the menu, choose what you want, and then the waiter takes your order to the kitchen and brings your food back to you.

In the same way, an API is a set of rules that allows different software applications to communicate with each other. It acts as a middleman between different pieces of software, letting them share information and work together without needing to know the details of how each one is built.

For example, when you use an app on your phone to check the weather, that app uses an API to ask another service for the weather data. The API sends your request to the weather service, gets the information you need, and then sends it back to the app, which displays it for you to see.

So, in short, an API helps different software talk to each other, making it easier for us to use various services and features without needing to understand all the technical details behind them.

---
## 4. Zero-Shot Classification

LLMs are excellent **zero-shot classifiers**. No training data needed!

In [10]:
texts = [
    "I absolutely love this product! Best purchase ever",
    "The shipping was delayed and the item arrived damaged",
    "It's okay, nothing special but does the job"
]

print(f"{'Text':<55} {'Sentiment'}")
print("_" * 101)

for text in texts:
    prompt = f"Classify the sentiment as exactly one word: positive, negative or neutral\n\nText: {text}\n\nClassification:"
    sentiment = ask(prompt, temperature=0.0, max_resp_tokens=10).strip().lower()
    emoji = {"positive": "🟢", "negative": "🔴", "neutral": "🟡"}.get(sentiment, "⚪")
    print(f"{text[:52]+'...':<55} {emoji} {sentiment}")

Text                                                    Sentiment
_____________________________________________________________________________________________________
I absolutely love this product! Best purchase ever...   🟢 positive
The shipping was delayed and the item arrived damage... 🔴 negative
It's okay, nothing special but does the job...          🟡 neutral


# Role - Context -Structure

In [11]:
prompt = "Explain what an API is to 10 years old child. Do it in 3 short sentences."
system = "You are my school teacher who loves analogies and explain everything in easy way"

result = ask(prompt, system_content=system)

display(Markdown(f"### Technical writer:\n\n{result}"))

### Technical writer:

An API is like a waiter in a restaurant. When you want food, you tell the waiter what you want, and they bring it to you from the kitchen. In the same way, an API helps different computer programs talk to each other and share information easily!

In [12]:
prompt = "Explain what an API is to 10 years old child. Do it in 3 short sentences. Put them in ordered bullets"
system = "You are my school teacher who loves analogies and explain everything in easy way"

result = ask(prompt, system_content=system)

display(Markdown(f"### Technical writer:\n\n{result}"))

### Technical writer:

Sure! Here’s a simple way to understand an API:

1. Imagine you go to a restaurant, and you tell the waiter what food you want from the menu.  
2. The waiter takes your order to the kitchen and brings your food back to you.  
3. An API is like that waiter; it helps different computer programs talk to each other and get what they need!

> 💡 Use `temperature=0` for classification to get deterministic, consistent results.

---
## 5. Structured Output (JSON)

In [13]:
import json

prompt = """Extract information from this text and return as valid JSON:

TEXT: "Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss the Q1 budget report."

Return JSON fields: attendee, date, time, topic

JSON:
"""

result = ask(prompt, temperature=0.0)
try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

Raw output: ```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```
Not valid JSON


### Below a better prompt to give only the Raw JSON

In [14]:
prompt = """Extract information from this text and return as valid JSON:

TEXT: "Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss the Q1 budget report."

Return JSON fields: attendee, date, time, topic

Do NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON objects, nothing else.

JSON:
"""

result = ask(prompt, temperature=0.0)
try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [15]:
prompt = """Extract information from this text and return as valid JSON:

TEXT: "Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss the Q1 budget report."

Return JSON fields: attendee, date, time, topic

JSON:
"""

response_json = client.chat.completions.create(
    model= OPENAI_MODEL,
    messages=[{'role': 'user', 'content': prompt}],
    response_format={"type": "json_object"}, # force valid JSON output
    temperature= 0.0
)

result = response_json.choices[0].message.content

try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [16]:
prompt = "Return a short greeting and a lucky number."
response_json = client.chat.completions.create(
    model= OPENAI_MODEL,
    messages=[{'role': 'user', 'content': prompt}],
    response_format={"type": "json_schema",
                     'json_schema': {
                         'name':"greeting_response",
                         'schema': {
                             "type": "object",
                             "properties": {
                                 "greeting": {"type": "string"},
                                 "luckyNumber": {"type": "integer"}
                             },
                             "required": ["greeting", "luckyNumber"],
                             "additionalProperties": False
                         },
                         "strict": True
                     }}, 
    temperature= 0.0
)

result = response_json.choices[0].message.content

try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

```json
{
  "greeting": "Hello! Wishing you a wonderful day!",
  "luckyNumber": 7
}
```

In [17]:
import json
 
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Describe an API endpoint for user login."}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "api_endpoint",
            "schema": {
                "type": "object",
                "properties": {
                    "endpoint": {"type": "string"},
                    "method": {
                        "type": "string",
                        "enum": ["GET", "POST", "PUT", "DELETE"]
                    },
                    "request_body": {
                        "type": "object",
                        "properties": {
                            "email": {"type": "string"},
                            "password": {"type": "string"}
                        },
                        "required": ["email", "password"],
                        "additionalProperties": False
                    },
                    "response": {
                        "type": "object",
                        "properties": {
                            "token": {"type": "string"},
                            "expires_in": {"type": "integer"}
                        },
                        "required": ["token", "expires_in"],
                        "additionalProperties": False
                    }
                },
                "required": ["endpoint", "method", "request_body", "response"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    temperature=0
)
 
parsed = json.loads(response.choices[0].message.content)
print(json.dumps(parsed, indent=2))

{
  "endpoint": "/api/login",
  "method": "POST",
  "request_body": {
    "email": "user@example.com",
    "password": "yourpassword"
  },
  "response": {
    "token": "abc123xyz",
    "expires_in": 3600
  }
}


---
## 6. Constraints

Adding explicit **constraints** controls length, format, and content:

In [4]:
prompt = """Write a product description for a wireless mouse.

Constrains:
- Maximum 50 words.
- Include at least one benefit.
- Do not mention price.
- End with a call to action.

Descreption:"""

result = ask(prompt)
word_count = len(result.split())

display(Markdown(f"> {result}"))
print(f"words: {word_count}")

> Experience seamless navigation with our sleek wireless mouse, designed for comfort and precision. Enjoy the freedom of movement with its advanced wireless technology, enhancing your productivity without the hassle of tangled cords. Elevate your workspace today—grab yours now and transform the way you work!

words: 44


### Multi-dimentional constraints (style - structure - semantics)

In [5]:
prompt = """01_zero_shot_basics.ipynbWrite a product description for a wireless mouse.

Constraints:
- 40-60 words.
- Exactly 2 sentences.
- First sentence: describe features.
- Second sentence: emphasize a user benefit.
- Incluse exactly 1 emoji.
- Must contain the word "precision"
- Do not use passive voice.
- End with a call to action.

Return Only the description."""

result = ask(prompt)
word_count = len(result.split())

display(Markdown(f"> {result}"))
print(f"words: {word_count}")

> Experience seamless navigation with our wireless mouse, featuring ergonomic design, adjustable DPI settings for precision, and long-lasting battery life. Enhance your productivity and comfort today—grab yours now! 🖱️

words: 28


---
## 7. Prompt Gallery: Real-World System Prompts

Let's study system prompts from **real community projects**. Each uses a different technique
to get reliable, high-quality outputs.

| Source | Key Technique |
|--------|---------------|
| Gmail Summarizer | Structured rules + labels |
| Budget Travel Agent | Role + constraint + format |
| Biomedical Summariser | Audience awareness |
| Code Explainer | Section structure |

In [6]:
# ── Prompt Gallery: 4 real-world system prompts ─────────
 
gallery = {
    "Gmail Summarizer": {
        "prompt": """You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.""",
        "technique": "Structured rules with labels and constraints",
    },
    "Budget Travel Agent": {
        "prompt": """You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.""",
        "technique": "Role + numbered constraints + format (markdown)",
    },
    "Biomedical Summariser": {
        "prompt": """Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.""",
        "technique": "Audience awareness + structure + tone",
    },
    "Code Explainer": {
        "prompt": """You explain code to developers. Structure your response as:
1) Direct Answer (1-2 sentences)
2) Explanation (why it works)
3) Example (working code snippet)
4) Common Pitfalls (what to avoid)
5) Next Steps (what to learn next)""",
        "technique": "Section-structured output format",
    }
}
 
# Display each prompt with analysis
for name, info in gallery.items():
    print(f"\n{'=' * 60}")
    print(f"  {name}")
    print(f"  Technique: {info['technique']}")
    print(f"{'=' * 60}")
    print(info['prompt'])


  Gmail Summarizer
  Technique: Structured rules with labels and constraints
You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.

  Budget Travel Agent
  Technique: Role + numbered constraints + format (markdown)
You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.

  Biomedical Summariser
  Technique: Audience awareness + structure + tone
Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.

  Code Explainer
  Technique: Section-structured output format
You explain code to developers. 

In [7]:
travel_result = ask(
    prompt="I'am visiting Sofia, Bulgaria for 3 days on a tight budget.",
    system_content= gallery["Budget Travel Agent"]["prompt"]
)

display(Markdown(f"### Budget Travel Agent Response\n\n{travel_result}"))

### Budget Travel Agent Response

# Budget Travel Guide to Sofia, Bulgaria

## Top 5 FREE Attractions
1. **Alexander Nevsky Cathedral**  
   A stunning Orthodox cathedral known for its impressive architecture and beautiful domes.

2. **Vitosha Boulevard**  
   The main commercial street in Sofia, perfect for a leisurely stroll, people-watching, and enjoying street performances.

3. **The National Palace of Culture (NDK)**  
   A large cultural complex surrounded by beautiful gardens, where you can enjoy the outdoor space and see various public art installations.

4. **Sofia's Parks**  
   Spend time in one of Sofia's many parks, such as Borisova Gradina or South Park, great for relaxation and picnics.

5. **The Roman Ruins of Serdica**  
   Explore the ancient ruins located in the heart of the city, offering a glimpse into Sofia's rich history.

## Budget Restaurants (Under $15/Meal)
1. **Sasa Asian Pub**  
   A casual dining spot offering a variety of Asian dishes and sushi at affordable prices.

2. **Made in Home**  
   A cozy restaurant with a homey atmosphere, serving delicious Bulgarian and international dishes at reasonable prices.

3. **Hadjidraganov's Cellars**  
   A traditional Bulgarian restaurant where you can try local dishes and enjoy a lively atmosphere without breaking the bank.

## Money-Saving Local Tip
- **Use Public Transport**: Sofia has an extensive and efficient public transport system, including trams, buses, and the metro. Purchase a day pass for unlimited travel, which is much cheaper than taxis and allows you to explore the city easily.

In [9]:
gmail_result = ask(
    """Here's an email thread:
From: marketing@company.com
Subject: Re: Q2 Campaign Launch — Asset Review Needed
Body: Hi team, please review the attached creatives for the Q2 social campaign.
We need approvals by Friday. The campaign landing page is at https://company.com/q2launch.
Let me know if any changes are needed. Thanks!""",
    system_content=gallery["Gmail Summarizer"]["prompt"]
)
display(Markdown(f"### Gmail Summarizer Response\n\n{gmail_result}"))
print("\nNotice the consistent labeling and structured bullet format!")

### Gmail Summarizer Response

1. Subject line: Q2 Campaign Launch — Asset Review Needed  
   Label: ACTION_REQUIRED  
   Summary: The marketing team requests a review of attached creatives for the Q2 social campaign. Approvals are needed by Friday, and the campaign landing page is provided.  
   Has link: yes


Notice the consistent labeling and structured bullet format!


In [10]:
bio_result = ask(
    """A 2024 study published in The Lancet examined the efficacy of a novel mRNA-based
therapeutic vaccine for stage III melanoma. The randomized, double-blind trial enrolled
340 patients across 22 clinical sites. Results showed a 44% reduction in recurrence risk
(HR 0.56, 95% CI 0.40–0.78, p=0.0007) over a 24-month follow-up. Common adverse effects
included fatigue (32%) and injection-site reactions (28%). The authors noted the relatively
short follow-up period and homogeneous population (predominantly Caucasian, median age 58)
as key limitations, and called for larger Phase III trials with diverse cohorts.""",
    system_content=gallery["Biomedical Summariser"]["prompt"]
)
display(Markdown(f"### Biomedical Summariser Response\n\n{bio_result}"))
print("\nNotice how it highlights methodology, limitations, and stays accessible!")

### Biomedical Summariser Response

**Key Findings:**
- The study evaluated a novel mRNA-based therapeutic vaccine for patients with stage III melanoma.
- Results demonstrated a 44% reduction in the risk of recurrence over a 24-month period (Hazard Ratio 0.56, 95% Confidence Interval 0.40–0.78, p=0.0007).
- Common adverse effects reported were fatigue (32%) and injection-site reactions (28%).

**Methodology:**
- The study was a randomized, double-blind trial.
- A total of 340 patients were enrolled from 22 clinical sites.

**Limitations:**
- The follow-up period of 24 months was relatively short.
- The study population was homogeneous, predominantly Caucasian with a median age of 58, which may not reflect broader demographics.

**Future Directions:**
- The authors recommend conducting larger Phase III trials with more diverse patient cohorts to validate these findings and assess long-term efficacy and safety.


Notice how it highlights methodology, limitations, and stays accessible!


In [12]:
code_result = ask(
    """Explain this Python code:
result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}""",
    system_content=gallery["Code Explainer"]["prompt"]
)
display(Markdown(f"### Code Explainer Response\n\n{code_result}"))
print("\nNotice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!")

### Code Explainer Response

1) Direct Answer: This code creates a dictionary `result` containing the top 5 key-value pairs from the `data` dictionary, sorted by their values in descending order.

2) Explanation: The `sorted()` function sorts the items of `data` based on the values (the second item of each key-value pair) using a lambda function as the key. The `reverse=True` argument sorts the values in descending order. The list slicing `[:5]` retrieves the top 5 items, and the dictionary comprehension constructs a new dictionary from these items.

3) Example:
```python
data = {
    'apple': 10,
    'banana': 5,
    'orange': 7,
    'grape': 15,
    'kiwi': 2,
    'mango': 12
}

result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}
print(result)  # Output: {'grape': 15, 'mango': 12, 'apple': 10, 'orange': 7, 'banana': 5}
```

4) Common Pitfalls: Ensure that `data` is a dictionary; otherwise, the method will throw an error. Additionally, be cautious with datasets that have fewer than five items, as the slicing may not return any results if not handled properly.

5) Next Steps: Learn about more advanced sorting techniques and how to handle cases where multiple items have the same value, which may affect the output in scenarios where ties are present.


Notice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!


> **Exercise:** Pick a business domain you're interested in (e.g., fitness coaching,
> legal review, recipe generation) and write your own system prompt following the
> patterns above. Test it with 3 different user queries.

---
## Key Takeaways 📝

| Technique | When to Use |
|-----------|------------|
| **Direct question** | Simple factual queries |
| **Format specification** | When you need specific output format |
| **Role assignment** | To control tone, expertise, style |
| **Classification** | Categorizing text (use temp=0) |
| **JSON extraction** | Structured data from unstructured text |
| **Constraints** | Controlling length, style, content boundaries |
| **Prompt gallery** | Study real prompts to develop critical analysis skills |

### Zero-Shot Tips
1. Be **specific** about what you want
2. Specify the **output format** explicitly
3. Use **roles/personas** to guide behaviour
4. Add **negative constraints** (what NOT to do)
5. Use `temperature=0` for deterministic tasks

---
**Next:** `02_few_shot_examples.ipynb` — Improve results by providing examples